# Final Test Evaluation — FOML AI vs Real Image Detector

This notebook performs the **one-time final evaluation** of our already-trained classical ML models and CNN on the reserved `test` split.

## Upload these three files when prompted
1. `split_foml_dataset.zip`
2. `classical_ml_models.joblib`
3. `cnn_baseline_artifacts.zip`

It does **not** train or tune models. It loads the saved models and evaluates each on the same 30 test images. This keeps the comparison fair and avoids using the test set during model development.

**Important:** Once we inspect these test results, do not keep changing models based on this same test set and then report it as an untouched final test. Any further development should use the validation set; for another unbiased final test, collect a new independent test set.


In [ ]:
# Install the scikit-learn version used when our classical models were saved.
# This helps avoid version-compatibility warnings when loading the saved PCA/classifiers.
%pip -q install scikit-learn==1.6.1

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import joblib
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

## 1. Upload the saved dataset and model files

Upload the same split ZIP used for the earlier experiments, the classical model bundle, and the CNN artifact ZIP. Do not upload a different dataset split.

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_names = list(uploaded.keys())

dataset_candidates = [n for n in uploaded_names if n.lower().endswith(".zip") and "split_foml_dataset" in n.lower()]
cnn_candidates = [n for n in uploaded_names if n.lower().endswith(".zip") and "cnn_baseline_artifacts" in n.lower()]
classical_candidates = [n for n in uploaded_names if n.lower().endswith(".joblib") and "classical_ml_models" in n.lower()]

if len(dataset_candidates) != 1:
    raise ValueError("Upload exactly one split_foml_dataset.zip.")
if len(cnn_candidates) != 1:
    raise ValueError("Upload exactly one cnn_baseline_artifacts.zip.")
if len(classical_candidates) != 1:
    raise ValueError("Upload exactly one classical_ml_models.joblib.")

dataset_zip = Path("/content") / dataset_candidates[0]
cnn_zip = Path("/content") / cnn_candidates[0]
classical_path = Path("/content") / classical_candidates[0]

print("Dataset:", dataset_zip.name)
print("CNN artifacts:", cnn_zip.name)
print("Classical model bundle:", classical_path.name)

## 2. Extract and verify the test split

We will only read images from `test/`. The training and validation folders are not used in this notebook.

In [ ]:
WORK_DIR = Path("/content/foml_final_test_evaluation")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

dataset_extract = WORK_DIR / "dataset"
cnn_extract = WORK_DIR / "cnn_artifacts"
dataset_extract.mkdir()
cnn_extract.mkdir()

with zipfile.ZipFile(dataset_zip, "r") as zf:
    zf.extractall(dataset_extract)

with zipfile.ZipFile(cnn_zip, "r") as zf:
    zf.extractall(cnn_extract)

metadata_files = list(dataset_extract.rglob("split_metadata.csv"))
if not metadata_files:
    raise FileNotFoundError("Could not find split_metadata.csv in the dataset ZIP.")
dataset_root = metadata_files[0].parent

cnn_model_paths = list(cnn_extract.rglob("cnn_model.keras"))
if not cnn_model_paths:
    raise FileNotFoundError("Could not find cnn_model.keras in the CNN ZIP.")
cnn_model_path = cnn_model_paths[0]

test_root = dataset_root / "test"
if not test_root.exists():
    raise FileNotFoundError(f"Test folder not found: {test_root}")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LABEL_TO_NUMBER = {"real": 0, "ai_generated": 1}
NUMBER_TO_LABEL = {0: "Real", 1: "AI-generated"}

# The fixed order ensures class labels remain consistent across all models.
test_records = []
for label_name, label_id in LABEL_TO_NUMBER.items():
    label_folder = test_root / label_name
    if not label_folder.exists():
        raise FileNotFoundError(f"Missing test class folder: {label_folder}")
    paths = sorted(
        p for p in label_folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    for image_path in paths:
        # Category folder is directly above each image in this dataset.
        category = image_path.parent.name
        test_records.append({
            "path": image_path,
            "filename": image_path.name,
            "category": category,
            "label_name": label_name,
            "label_id": label_id,
        })

test_counts = pd.Series([r["label_name"] for r in test_records]).value_counts().to_dict()
display(pd.DataFrame([
    {"label": label, "count": test_counts.get(label, 0)}
    for label in ["real", "ai_generated"]
]))

print("Total test images:", len(test_records))
if len(test_records) != 30 or test_counts.get("real", 0) != 15 or test_counts.get("ai_generated", 0) != 15:
    print("WARNING: The test split differs from the expected 30 images (15 per class). Please check the ZIP and split summary.")

## 3. Load the saved classical models and CNN

The classical bundle contains the PCA transformation and the four classifiers trained earlier. The CNN ZIP contains the trained CNN model. We load them as-is; we do not fit anything again.

In [ ]:
classical_bundle = joblib.load(classical_path)
pca = classical_bundle["pca"]
classical_models = classical_bundle["models"]
classical_image_size = tuple(classical_bundle["image_size"])

cnn_model = tf.keras.models.load_model(cnn_model_path, compile=False)

print("Classical image size:", classical_image_size)
print("Classical models:", list(classical_models.keys()))
print("CNN input shape:", cnn_model.input_shape)
print("CNN loaded successfully.")

## 4. Evaluate classical ML models

For the classical pipeline we repeat the original feature extraction: resize to 32 × 32, convert to RGB, scale pixel values to 0–1, flatten the pixels, and apply the PCA fitted on the training data. The PCA and classifiers are not refitted.

In [ ]:
def image_to_classical_features(image_path, size):
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        image = image.resize(size)
        array = np.asarray(image, dtype=np.float32) / 255.0
    return array.reshape(-1)

y_test = np.array([record["label_id"] for record in test_records], dtype=np.int64)
X_test_classical = np.stack([
    image_to_classical_features(record["path"], classical_image_size)
    for record in test_records
])
X_test_pca = pca.transform(X_test_classical)

print("Classical test feature shape:", X_test_classical.shape)
print("PCA-transformed test feature shape:", X_test_pca.shape)
print("Test labels (0=Real, 1=AI-generated):", y_test.shape)

## 5. Evaluate the CNN on the same test images

The CNN was trained at 256 × 256 pixels. We load each image at that size with bilinear interpolation, matching the earlier Keras image-loading setup. The model itself contains the 1/255 rescaling layer.

In [ ]:
CNN_IMAGE_SIZE = (256, 256)

X_test_cnn = []
for record in test_records:
    image = tf.keras.utils.load_img(
        record["path"],
        target_size=CNN_IMAGE_SIZE,
        color_mode="rgb",
        interpolation="bilinear",
    )
    X_test_cnn.append(tf.keras.utils.img_to_array(image))

X_test_cnn = np.stack(X_test_cnn).astype(np.float32)

cnn_probabilities_ai = cnn_model.predict(X_test_cnn, verbose=0).reshape(-1)
cnn_predictions = (cnn_probabilities_ai >= 0.5).astype(int)

print("CNN test array shape:", X_test_cnn.shape)
print("CNN predictions generated:", len(cnn_predictions))

## 6. Compare models with the held-out test set

We report accuracy and the precision, recall, and F1-score for the AI-generated class. These metrics complement each other: accuracy shows overall correctness, while AI precision and recall show different types of errors.

In [ ]:
test_results = []
all_predictions = {}

for model_name, model in classical_models.items():
    predictions = model.predict(X_test_pca).astype(int)
    all_predictions[model_name] = predictions

    test_results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision_ai": precision_score(y_test, predictions, pos_label=1, zero_division=0),
        "recall_ai": recall_score(y_test, predictions, pos_label=1, zero_division=0),
        "f1_ai": f1_score(y_test, predictions, pos_label=1, zero_division=0),
        "test_images": len(y_test),
    })

all_predictions["Small CNN (trained from scratch)"] = cnn_predictions
test_results.append({
    "model": "Small CNN (trained from scratch)",
    "accuracy": accuracy_score(y_test, cnn_predictions),
    "precision_ai": precision_score(y_test, cnn_predictions, pos_label=1, zero_division=0),
    "recall_ai": recall_score(y_test, cnn_predictions, pos_label=1, zero_division=0),
    "f1_ai": f1_score(y_test, cnn_predictions, pos_label=1, zero_division=0),
    "test_images": len(y_test),
})

test_results_df = pd.DataFrame(test_results).sort_values("f1_ai", ascending=False).reset_index(drop=True)
display(test_results_df.round(3))

print("\nDetailed classification reports:")
for model_name, predictions in all_predictions.items():
    print("\n---", model_name, "---")
    print(classification_report(
        y_test, predictions,
        labels=[0, 1],
        target_names=["Real", "AI-generated"],
        zero_division=0,
    ))

## 7. Confusion matrices and per-image predictions

Each confusion matrix uses rows for the actual label and columns for the predicted label. The off-diagonal cells are mistakes.

Because the test set is small, treat each metric as an estimate rather than proof of general performance. With 30 images, one image corresponds to about 3.3 percentage points in accuracy.

In [ ]:
num_models = len(all_predictions)
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, (model_name, predictions) in zip(axes, all_predictions.items()):
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])
    display_obj = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Real", "AI-generated"],
    )
    display_obj.plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(model_name)

for ax in axes[num_models:]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(WORK_DIR / "test_confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()

prediction_table = pd.DataFrame({
    "filename": [r["filename"] for r in test_records],
    "category": [r["category"] for r in test_records],
    "actual_label": [NUMBER_TO_LABEL[r["label_id"]] for r in test_records],
})
for model_name, predictions in all_predictions.items():
    prediction_table[model_name + "_prediction"] = [
        NUMBER_TO_LABEL[int(value)] for value in predictions
    ]

display(prediction_table.head(10))

## 8. Save the final test report

These files are the evaluation outputs, not a newly trained model. Keep them with the project results and use the same metrics in the report.

In [ ]:
results_csv = WORK_DIR / "final_test_results.csv"
predictions_csv = WORK_DIR / "final_test_predictions.csv"
report_zip = Path("/content/foml_final_test_report.zip")

test_results_df.to_csv(results_csv, index=False)
prediction_table.to_csv(predictions_csv, index=False)

with zipfile.ZipFile(report_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(results_csv, arcname=results_csv.name)
    zf.write(predictions_csv, arcname=predictions_csv.name)
    zf.write(WORK_DIR / "test_confusion_matrices.png", arcname="test_confusion_matrices.png")

print("Saved report:", report_zip)
print("Reminder: do not tune models using these test results and then report the same test set as untouched.")
files.download(str(report_zip))